**Step 1: Dataset Preparation and Exploratory Data Analysis (EDA)**
In this step, we aim to:

* Load the Banking77 dataset.

* Understand the distribution of intents.

* Explore sentence lengths and vocabulary size.

* Prepare the text for tokenization using the DistilBERT tokenizer.

Understanding the dataset and vocabulary is critical before we build a model. It helps us identify preprocessing requirements, validate class balance, and choose model hyperparameters such as sequence length and embedding dimensions.

**Step 1.1: Install Required Libraries**

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.5 MB/s eta 0:00:00
   ━━

In [ ]:
!pip install --upgrade datasets==2.16.0 huggingface_hub transformers

 Using cached datasets-2.16.0-py3-none-any.whl.metadata (20 kB)
 Using cached huggingface_hub-0.31.4-py3-none-any.whl.metadata (13 kB)
 Using cached transformers-4.52.3-py3-none-any.whl.metadata (40 kB)
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 11.2 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.3/489.3 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 20.0 MB/s eta 0:00:00
 Attempting uninstall: fsspec
 Found existing installation: fsspec 2025.3.2
 Uninstalling fsspec-2025.3.2:
 Successfully uninstalled fsspec-2025.3.2
 Attempting uninstall: huggingface_hub
 Found existing installation: huggingface-hub 0.31.2
 Uninstalling huggingface-hub-0.31.2:
 Successfully uninstalled huggingface-hub-0.31.2
ERROR: Operation cancelled by user
^C


In [ ]:
from datasets import Dataset, DatasetDict, Features, ClassLabel, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
import os
import faiss
from sentence_transformers import SentenceTransformer
import pickle

# Load the dataset
dataset = DatasetDict.load_from_disk("/content/banking77_dataset")

# Preprocessing function
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def preprocess_function(examples):
 return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

# Apply preprocessing
custom_dataset = dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
custom_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# Train base model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=dataset["train"].features["label"].num_classes
).to(device)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=custom_dataset["train"],
    eval_dataset=custom_dataset["validation"],
    tokenizer=tokenizer,
)

trainer.train()


# Evaluate base model
eval_results = trainer.evaluate(custom_dataset["test"])
print(f"Base Model Evaluation: {eval_results}")

test_preds = trainer.predict(custom_dataset["test"]).predictions.argmax(axis=1)
test_labels = custom_dataset["test"]["label"]

misclassified = [(pred, true, custom_dataset['test'][i]['input_ids']) for i, (pred, true) in enumerate(zip(test_preds, test_labels)) if pred != true]
for pred, true, input_ids in misclassified[:5]:
 text = tokenizer.decode(input_ids, skip_special_tokens=True)
 print(f"Text: {text}, Predicted: {pred}, True: {true}")


# Function to create FAISS index
def create_faiss_index(dataset, model_name='BAAI/bge-large-en-v1.5'):
    print(f"Loading SentenceTransformer: {model_name}")
    embedder = SentenceTransformer(model_name).to(device)
    print("Encoding FAISS texts...")
    # Retrieve original texts from the dataset before tokenization if available,
    if 'text' in dataset['faiss'].features:
        texts = dataset['faiss']['text']
    else:
        texts = [tokenizer.decode(example['input_ids'], skip_special_tokens=True) for example in dataset['faiss']]

    embeddings = embedder.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

    print("Building FAISS index...")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(index, 'faiss_index_sentence_transformer.idx')
    with open('faiss_texts_sentence_transformer.pkl', 'wb') as f:
        pickle.dump(texts, f)
    return index, texts, embedder

# Create FAISS index
faiss_index, faiss_texts, embedder = create_faiss_index(dataset)


# RAG Inference Function
def rag_inference(query, model, tokenizer, faiss_index, faiss_texts, embedder, k=3, max_length=128, distance_threshold=0.7, num_predictions=5):
    # 1. Embed the query
    query_embedding = embedder.encode([query], convert_to_numpy=True)

    # 2. Retrieve top-k similar examples from FAISS
    D, I = faiss_index.search(query_embedding, k) # D: distances, I: indices
    retrieved_indices = I[0] # Get indices for the first (and only) query

    # Filter based on distance threshold
    filtered_retrieved_indices = [idx for i, idx in enumerate(retrieved_indices) if D[0][i] <= distance_threshold]

    if not filtered_retrieved_indices:
        # Fallback to base model if no relevant examples are found
        # print("No relevant examples found via FAISS. Falling back to base model.")
        inputs = tokenizer(query, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length).to(model.device)
        with torch.no_grad():
            logits = model(**inputs).logits
        return torch.argmax(logits, dim=1).cpu().numpy()[0]

    # 3. Construct context from retrieved examples
    context_texts = [faiss_texts[i] for i in filtered_retrieved_indices]
    combined_input = query + " [SEP] " + " ".join(context_texts)

    # 4. Tokenize and get prediction from the model
    inputs = tokenizer(combined_input, return_tensors="pt", truncation=True, padding="max_length", max_length=max_length).to(model.device)

    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Sort predictions by confidence and return top N
    probabilities = torch.softmax(logits, dim=-1)
    top_n_probabilities, top_n_indices = torch.topk(probabilities, k=num_predictions, dim=-1)
    
    # Return the ID of the top prediction
    return top_n_indices[0][0].cpu().numpy()


# Fine-tuning with Optuna (Hyperparameter Optimization)
import optuna

def objective(trial):
    # Hyperparameters to tune for RAG
    k = trial.suggest_int('k', 1, 10) # Number of retrieved examples
    distance_threshold = trial.suggest_float('distance_threshold', 0.1, 0.9) # FAISS distance threshold
    max_rag_length = trial.suggest_int('max_rag_length', 64, 256) # Max token length for RAG input
    num_predictions = trial.suggest_int('num_predictions', 1, 77) # Number of top predictions to consider for reranking

    correct_predictions = 0
    total_predictions = 0

    for i, example in enumerate(custom_dataset['validation']):
        query_text = tokenizer.decode(example['input_ids'], skip_special_tokens=True)
        true_label = example['label'].item() # Convert tensor to Python int

        # Get RAG prediction
        # The rag_inference function returns a numpy array with a single element for the top prediction
        # So, we need to extract that single element for comparison
        predicted_label_id = rag_inference(query_text, model, tokenizer, faiss_index, faiss_texts, embedder,
                                           k=k, max_length=max_rag_length, distance_threshold=distance_threshold,
                                           num_predictions=num_predictions).item() # Extract the scalar value

        if predicted_label_id == true_label:
            correct_predictions += 1
        total_predictions += 1

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    return accuracy

# Create an Optuna study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Run 50 trials

print("Best trial:")
print("  Value: ", study.best_value)
print("  Params: ", study.best_params)

# Save best parameters
best_rag_params_path = "/content/optuna_results"
os.makedirs(best_rag_params_path, exist_ok=True)
with open(os.path.join(best_rag_params_path, "best_rag_params.json"), 'w') as f:
    json.dump(study.best_params, f, indent=4)


# Evaluate RAG model on test set with best parameters
print("\n--- Evaluating RAG Model on Test Set with Best Parameters ---\n")

best_k = study.best_params['k']
best_distance_threshold = study.best_params['distance_threshold']
best_max_rag_length = study.best_params['max_rag_length']
best_num_predictions = study.best_params['num_predictions']

correct_rag_predictions = 0
total_rag_predictions = 0

for i, example in enumerate(custom_dataset['test']):
    query_text = tokenizer.decode(example['input_ids'], skip_special_tokens=True)
    true_label = example['label'].item()

    rag_predicted_label_id = rag_inference(query_text, model, tokenizer, faiss_index, faiss_texts, embedder,
                                       k=best_k, max_length=best_max_rag_length, distance_threshold=best_distance_threshold,
                                       num_predictions=best_num_predictions).item()

    if rag_predicted_label_id == true_label:
        correct_rag_predictions += 1
    total_rag_predictions += 1

rag_accuracy = correct_rag_predictions / total_rag_predictions if total_rag_predictions > 0 else 0
print(f"RAG Model Test Accuracy: {rag_accuracy}")

### STEP8: Example Cases Where RAG Fixed Base Model Errors

This cell demonstrates **qualitative evidence** of how the RAG-enhanced model improves over the base MiniLLM intent classifier.

---

#### What It Does:

1. Iterates over the test set examples.
2. For each test query:
 - Gets the **base model's prediction**.
 - Gets the **RAG-enhanced prediction** using retrieval and reranking.
3. Compares predictions:
 - Finds cases where the base model **failed** (incorrect prediction),
 - But the **RAG-enhanced model corrected** the prediction (matched the true intent).
4. Stops after showing **5 successful correction cases**.

---

#### Output Format:

- **Query Text**: The user query from the test set.
- **True Label**: Ground-truth intent (in human-readable form).
- **Base Model Prediction**: Incorrect intent predicted by the MiniLLM.
- **RAG Prediction**: Corrected prediction by the RAG model.

This cell helps to **surface "win cases"** for RAG that are not easily visible through numeric metrics alone. These real examples are especially useful for:
- Stakeholder presentations
- Debugging model behavior
- Justifying improvements from retrieval-augmented methods


In [ ]:
# Save PyTorch model
torch.save(model.state_dict(), "final_rag_model.pth")

tokenizer.save_pretrained("final_tokenizer")
embedder.save("final_embedder")


In [ ]:
print("\n--- Showing 5 Cases Where RAG Fixed Base Model Errors (with Intent Names) ---\n")
base_wrong_rag_correct = []


test_texts = [tokenizer.decode(example['input_ids'], skip_special_tokens=True) for example in custom_dataset['test']]
labels = [example['label'] for example in custom_dataset['test']]


if 'correct_label_encoder' not in globals():
 from sklearn.preprocessing import LabelEncoder
 correct_label_encoder = LabelEncoder()
 label_names = dataset.features['label'].names
 correct_label_encoder.fit(label_names)
label_encoder = correct_label_encoder

for i, text in enumerate(test_texts):
 input_ids = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).input_ids.to(device)
 attention_mask = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).attention_mask.to(device)

 with torch.no_grad():
 base_logits = model(input_ids, attention_mask)
 base_pred_id = torch.argmax(base_logits, dim=1).cpu().numpy()[0]


 if 'best_k' not in globals():
 import json
 optuna_results_path = "/content/optuna_results/best_rag_params.json"
 if os.path.exists(optuna_results_path):
 with open(optuna_results_path, 'r') as f:
 best_params = json.load(f)
 best_k = best_params.get('k', 3) # Default to 3 if not found
 best_distance_threshold = best_params.get('distance_threshold', 0.7) # Default to 0.7
 best_max_rag_length = best_params.get('max_rag_length', 50) # Default to 50
 else:
 print("Warning: Optuna results not found. Using default RAG parameters.")
 best_k = 3
 best_distance_threshold = 0.7
 best_max_rag_length = 50


 rag_pred_id = rag_inference(text, model, tokenizer, faiss_index, faiss_texts, embedder,
 k=best_k, max_length=best_max_rag_length, distance_threshold=best_distance_threshold,
 num_predictions=25) # Use the same number of predictions as final eval

 true_label_id = labels[i] # Use the labels list defined above

 if base_pred_id != true_label_id and rag_pred_id == true_label_id:
 base_wrong_rag_correct.append({
 "text": text,
 "true_label": label_encoder.inverse_transform([true_label_id])[0],
 "base_pred": label_encoder.inverse_transform([base_pred_id])[0],
 "rag_pred": label_encoder.inverse_transform([rag_pred_id])[0]
 })

 if len(base_wrong_rag_correct) == 5:
 break

# Print results
for ex in base_wrong_rag_correct:
 print(f"Query Text: {ex['text']}")
 print(f"True Label: {ex['true_label']}")
 print(f"Base Model Prediction: {ex['base_pred']}")
 print(f"RAG Prediction: {ex['rag_pred']}")
 print("-" * 80)


--- Showing 5 Cases Where RAG Fixed Base Model Errors (with Intent Names) ---

Query Text: is there something wrong with the atm? it would not let me pull cash out of my account.
True Label: declined_card_payment
Base Model Prediction: wrong_amount_of_cash_received
RAG Prediction: declined_card_payment
--------------------------------------------------------------------------------
Query Text: do you charge extra for duplicate cards?
True Label: get_physical_card
Base Model Prediction: transaction_charged_twice
RAG Prediction: get_physical_card
--------------------------------------------------------------------------------
Query Text: i topped off my card is that something you will charge me for?
True Label: top_up_by_card_charge
Base Model Prediction: pending_cash_withdrawal
RAG Prediction: top_up_by_card_charge
--------------------------------------------------------------------------------
Query Text: why was my card payment reversed?
True Label: reverted_card_payment?
Base Model 

In [ ]:
import numpy as np
import torch
import os
from sklearn.preprocessing import LabelEncoder

def calculate_mrr(y_true, y_pred_ranked_lists):
 """
 Calculates the Mean Reciprocal Rank (MRR).

 Args:
 y_true (list or np.array): A list of ground truth relevant items for each query.
 Each element can be a single item or a list of relevant items.
 y_pred_ranked_lists (list of lists or np.array of lists): A list where each element is
 a ranked list of predicted/retrieved items for a query.

 Returns:
 float: The Mean Reciprocal Rank (MRR) score.
 """
 if not y_true or not y_pred_ranked_lists:
 return 0.0

 reciprocal_ranks = []
 for i, true_item in enumerate(y_true):
 predicted_list = y_pred_ranked_lists[i]

 if not isinstance(true_item, (list, np.ndarray)):
 true_item = [true_item]

 rank = 0
 found = False
 for r, pred_item in enumerate(predicted_list):
 if pred_item in true_item:
 rank = r + 1
 found = True
 break

 if found:
 reciprocal_ranks.append(1 / rank)
 else:
 reciprocal_ranks.append(0)

 return np.mean(reciprocal_ranks)



print("\n--- Generating RAG Predictions for MRR Calculation ---\n")

# Initialize lists to store all true labels and RAG predictions
all_true_labels = []
all_rag_predictions_ranked = []


test_texts = [tokenizer.decode(example['input_ids'], skip_special_tokens=True) for example in custom_dataset['test']]
labels = [example['label'] for example in custom_dataset['test']]


if 'correct_label_encoder' not in globals():
 label_names = dataset.features['label'].names
 correct_label_encoder = LabelEncoder()
 correct_label_encoder.fit(label_names)
label_encoder = correct_label_encoder

if 'best_k' not in globals():
 import json
 optuna_results_path = "/content/optuna_results/best_rag_params.json" # Ensure this path is correct in your env
 if os.path.exists(optuna_results_path):
 with open(optuna_results_path, 'r') as f:
 best_params = json.load(f)
 best_k = best_params.get('k', 3)
 best_distance_threshold = best_params.get('distance_threshold', 0.7)
 best_max_rag_length = best_params.get('max_rag_length', 50)
 else:
 print("Warning: Optuna results not found. Using default RAG parameters.")
 best_k = 3
 best_distance_threshold = 0.7
 best_max_rag_length = 50

for i, text in enumerate(test_texts):
 input_ids = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).input_ids.to(device)
 attention_mask = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).attention_mask.to(device)

 with torch.no_grad():
 base_logits = model(input_ids, attention_mask)
 base_pred_id = torch.argmax(base_logits, dim=1).cpu().numpy()[0]

 # RAG model prediction
 # Assuming rag_inference returns a single predicted ID for the top prediction
 rag_pred_id = rag_inference(query_text, model, tokenizer, faiss_index, faiss_texts, embedder,
 k=best_k, max_length=best_max_rag_length, distance_threshold=best_distance_threshold,
 num_predictions=25) # num_predictions determines the candidates, but rag_pred_id is single best here

 true_label_id = labels[i] # This is the true label ID

 # Convert IDs to label names using your label_encoder
 true_label_name = label_encoder.inverse_transform([true_label_id])[0]
 rag_pred_name = label_encoder.inverse_transform([rag_pred_id])[0] # Assuming rag_pred_id is a single ID

 # Append to the lists for MRR calculation
 all_true_labels.append(true_label_name)
 all_rag_predictions_ranked.append([rag_pred_name])


mrr_for_all_rag_predictions = calculate_mrr(all_true_labels, all_rag_predictions_ranked)

print(f"\nMRR for all RAG Predictions: {mrr_for_all_rag_predictions}")


--- Generating RAG Predictions for MRR Calculation ---


MRR for all RAG Predictions: 0.8667481662591687


**STEP 9: Save Final RAG Model and Assets for Deployment**

This cell saves all the essential components required to deploy the Retrieval-Augmented Generation (RAG)-based intent detection pipeline as an API.

---

#### Components Saved:

1. **MiniLLM Model Weights** 
 - `final_minillm_model.pth`: Trained model parameters using PyTorch.

2. **FAISS Index** 
 - `faiss_index.idx`: Precomputed dense index for fast nearest-neighbor search over support examples.

3. **FAISS Corpus Texts** 
 - `faiss_texts.pkl`: List of reference/support texts associated with FAISS embeddings.

4. **Embedding Model** 
 - `embedder.model`: The embedding model used to generate vector representations for retrieval.

5. **Tokenizer** 
 - `tokenizer/`: Directory containing tokenizer config and vocabulary, saved in HuggingFace format.

6. **Label Encoder** 
 - `label_encoder.pkl`: Used to map between intent labels (strings) and their corresponding indices.

---

#### Usage:

These files can now be loaded within a Flask, FastAPI, or other server framework to:
- Embed incoming user queries,
- Perform FAISS-based retrieval,
- Run classification using the RAG-enhanced model,
- Return predicted intents via an API response.

This setup supports **real-time intent detection** in production.


In [ ]:
save_dir = "final_rag_assets"
os.makedirs(save_dir, exist_ok=True)

# 1. Save the MiniLLM model
torch.save(model.state_dict(), os.path.join(save_dir, "final_minillm_model.pth"))

# 2. Save FAISS index
faiss.write_index(faiss_index, os.path.join(save_dir, "faiss_index.idx"))

# 3. Save FAISS corpus texts (list of strings)
with open(os.path.join(save_dir, "faiss_texts.pkl"), "wb") as f:
 pickle.dump(faiss_texts, f)

# 4. Save embedder
embedder.save(os.path.join(save_dir, "embedder.model"))

# 5. Save tokenizer
tokenizer.save_pretrained(os.path.join(save_dir, "tokenizer"))

# 6. Save LabelEncoder
with open(os.path.join(save_dir, "label_encoder.pkl"), "wb") as f:
 pickle.dump(label_encoder, f)

print(f"All RAG model components saved to '{save_dir}' for API deployment.")


All RAG model components saved to...